# Exploratory Notebook: Beyond Softmax

**ArXivist-generated exploration notebook**  
Paper: [arXiv:2509.24728](https://arxiv.org/abs/2509.24728) · Manenti & Alippi, ICML 2026

This notebook provides four visual explorations of the paper's core ideas:

1. **Loss landscape comparison** — reproduce Figure 1: catnat vs softmax optimization landscape
2. **FIM structure sweep** — how the FIM changes with K and activation choice
3. **Gumbel temperature ablation** — effect of τ on sample sharpness
4. **Node reach probabilities** — visualizing P(a_i) through the catnat tree

> For checkpointed models: load a checkpoint from `checkpoints/` and set `CHECKPOINT_PATH` below.

In [ ]:
import sys, os, math
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Optional: load a checkpoint
CHECKPOINT_PATH = None  # e.g. "../checkpoints/vae/best.pt"

from src.catnat import CatNat, SoftmaxParam, SparseMaxParam, build_parameterization
from src.catnat.activations import NaturalActivation, SigmoidActivation
from src.catnat.utils.tree_utils import BinaryTreeIndex
from src.catnat.samplers import GumbelSoftmaxSampler
print("Imports OK.")

## Visualization 1: Loss Landscape Comparison (Figure 1)

The paper's Figure 1 shows that the catnat cross-entropy landscape is smoother and more regular than softmax, yielding more direct gradient-descent trajectories.

We reproduce this for a 4-class ($K=4$) problem with a uniform target distribution, sweeping two scores over a 2D grid.

In [ ]:
try:
    K = 4
    GRID = 50
    s_range = torch.linspace(-5, 5, GRID)
    s1g, s2g = torch.meshgrid(s_range, s_range, indexing='ij')

    target = torch.full((1, K), 1.0 / K)   # uniform target

    smx    = SoftmaxParam()
    catnat = CatNat(K=K, activation='natural')

    L_smx    = torch.zeros(GRID, GRID)
    L_catnat = torch.zeros(GRID, GRID)

    for i in range(GRID):
        for j in range(GRID):
            # Softmax: K scores, fix last two to 0
            s_s = torch.zeros(1, K); s_s[0,0]=s1g[i,j]; s_s[0,1]=s2g[i,j]
            p_s = smx(s_s)
            L_smx[i,j] = -(target * p_s.clamp(1e-8).log()).sum().item()

            # CatNat: K-1 scores, fix last one to 0
            s_c = torch.zeros(1, K-1); s_c[0,0]=s1g[i,j]; s_c[0,1]=s2g[i,j]
            p_c = catnat(s_c)
            L_catnat[i,j] = -(target * p_c.clamp(1e-8).log()).sum().item()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    grid_np = s_range.numpy()

    for ax, L, title, color in [
        (axes[0], L_smx,    'Softmax (dense FIM — curved landscape)',  'Blues'),
        (axes[1], L_catnat, 'catnat-ν (diagonal FIM — flat landscape)', 'Oranges'),
    ]:
        cf = ax.contourf(grid_np, grid_np, L.numpy().T, levels=30, cmap=color)
        ax.contour( grid_np, grid_np, L.numpy().T, levels=15, colors='black', linewidths=0.4, alpha=0.5)
        plt.colorbar(cf, ax=ax, label='Cross-entropy loss')

        # Add gradient-descent arrows starting from the same point
        torch.manual_seed(7)
        for _ in range(4):
            # Random starting point in [-4,4]
            start = torch.rand(2) * 6 - 3
            traj  = [start.tolist()]
            lr    = 0.3
            s_t   = start.clone().requires_grad_(True)
            for __ in range(12):
                if title.startswith('Softmax'):
                    s_in = torch.zeros(1, K)
                    s_in[0,0] = s_t[0]; s_in[0,1] = s_t[1]
                    p = smx(s_in)
                else:
                    s_in = torch.zeros(1, K-1)
                    s_in[0,0] = s_t[0]; s_in[0,1] = s_t[1]
                    p = catnat(s_in)
                loss = -(target * p.clamp(1e-8).log()).sum()
                grad = torch.autograd.grad(loss, s_t)[0]
                s_t = (s_t - lr * grad).detach().requires_grad_(True)
                traj.append(s_t.detach().tolist())
            traj = np.array(traj)
            ax.plot(traj[:,0], traj[:,1], 'k-o', ms=3, lw=1.5, alpha=0.8)
            ax.plot(traj[0,0], traj[0,1], 'go', ms=7)
            ax.plot(traj[-1,0], traj[-1,1], 'r*', ms=10)

        ax.set_title(title, fontsize=11)
        ax.set_xlabel('s₁'); ax.set_ylabel('s₂')
        ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)

    plt.suptitle('Loss landscape: catnat (right) is smoother → more direct gradient paths (Figure 1)',
                 fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()

except Exception as e:
    print(f"[ERROR] {e}")
    raise

## Visualization 2: FIM Diagonal Entries Across K

Theorem 4.2 guarantees diagonal FIM for catnat for any $K$. Corollary 4.3 shows that with $\nu$, the entries equal $P(a_i) \cdot (\pi/A)^2$. Here we plot the diagonal entries for $K \in \{4, 8, 16\}$ and compare natural vs sigmoid activation.

In [ ]:
try:
    A = 2 * math.pi
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for ax, K in zip(axes, [4, 8, 16]):
        s0 = torch.zeros(1, K-1)   # s=0: all nodes in active region

        for act, color, label in [
            ('natural', 'tab:blue',   f'catnat-ν (Corollary 4.3: const × P(aᵢ))'),
            ('sigmoid', 'tab:orange', f'catnat-σ (Theorem 4.2: P(aᵢ)·σ(sᵢ)(1-σ(sᵢ)))'),
        ]:
            m = CatNat(K=K, activation=act, A=A, C=0.0)
            diag  = m.fim_diagonal(s0).squeeze(0).detach()        # [K-1]
            reach = m.node_reach_probs(s0).squeeze(0).detach()    # [K-1]

            node_ids = list(range(K-1))
            ax.bar([x + (0.3 if act=='sigmoid' else 0) for x in node_ids],
                   diag.numpy(), width=0.3, label=label, color=color, alpha=0.8)

        # Annotate with P(a_i) values
        m_nu = CatNat(K=K, activation='natural')
        reach_nu = m_nu.node_reach_probs(torch.zeros(1,K-1)).squeeze(0).detach()
        for i, r in enumerate(reach_nu):
            ax.text(i, 0.001, f'{r:.2f}', ha='center', fontsize=6, color='navy')

        ax.set_title(f'K={K}, s=0  (blue values = P(aᵢ))', fontsize=10)
        ax.set_xlabel('Internal node index')
        ax.set_ylabel('Diagonal FIM entry G_ii')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    plt.suptitle('Diagonal FIM entries by node: catnat-ν vs catnat-σ (at s=0)', fontsize=12)
    plt.tight_layout(); plt.show()
    print("catnat-ν: all bars same height within a K → entries depend only on P(aᵢ), not sᵢ (Corollary 4.3)")

except Exception as e:
    print(f"[ERROR] {e}")
    raise

## Visualization 3: Gumbel Temperature Ablation

The Gumbel temperature $\tau$ controls how closely the continuous relaxation approximates a one-hot sample. At $\tau \to 0$ the sample is essentially one-hot; at $\tau = 1$ it is diffuse. The paper anneals from $\tau=1$ to $\tau=0.5$ during VAE training (Appendix F.3).

In [ ]:
try:
    torch.manual_seed(42)
    K = 8
    N_SAMPLES = 500
    catnat = CatNat(K=K, activation='natural')
    gumbel = GumbelSoftmaxSampler()

    # Fixed log-probs from a peaked distribution (mode at k=2)
    s = torch.zeros(1, K-1); s[0, 1] = 3.0   # bias node 1
    log_p = catnat.log_prob(s).expand(N_SAMPLES, -1)   # [N, K]

    taus = [2.0, 1.0, 0.5, 0.1]
    fig, axes = plt.subplots(1, len(taus), figsize=(14, 3.5))

    for ax, tau in zip(axes, taus):
        C_soft = gumbel(log_p, tau=tau, hard=False).detach()   # [N, K]
        # Average sample (should concentrate around the mode)
        mean_sample = C_soft.mean(0).numpy()
        # Entropy of mean (lower = more peaked)
        ent = -(mean_sample * np.log(mean_sample.clip(1e-8))).sum()

        ax.bar(range(K), mean_sample, color='steelblue', alpha=0.85)
        ax.axhline(1/K, color='red', ls='--', alpha=0.5, label='Uniform 1/K')
        ax.set_title(f'τ = {tau}\n(entropy={ent:.3f})', fontsize=10)
        ax.set_xlabel('Category k')
        ax.set_ylim(0, 1)
        if ax == axes[0]: ax.set_ylabel('Mean sample value')
        ax.legend(fontsize=7)

    plt.suptitle(f'Gumbel-Softmax samples (N={N_SAMPLES}) at different temperatures\n'
                 'Lower τ → more peaked samples → closer to one-hot', fontsize=11)
    plt.tight_layout(); plt.show()

    # Annealing schedule visualization
    steps = np.arange(0, 200_000, 1000)
    taus_sched = [gumbel.anneal_temperature(int(s), 1.0, 0.5, 3e-5) for s in steps]
    fig2, ax2 = plt.subplots(figsize=(8, 3))
    ax2.plot(steps, taus_sched, lw=2, color='steelblue')
    ax2.axhline(0.5, color='red', ls='--', label='τ_min = 0.5')
    ax2.set_xlabel('Training step'); ax2.set_ylabel('Temperature τ')
    ax2.set_title('Gumbel temperature annealing schedule (Appendix F.3: rate=3e-5)')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

except Exception as e:
    print(f"[ERROR] {e}")
    raise

## Visualization 4: Node Reach Probabilities $P(a_i)$ Through the Tree

For catnat with $K=8$, there are 7 internal nodes arranged in a binary tree of depth $H=3$. The reach probability $P(a_i)$ — the probability of descending from the root to node $i$ — determines the scale of each node's diagonal FIM entry. We visualize how $P(a_i)$ distributes across the tree as scores vary.

In [ ]:
try:
    K = 8
    H = int(math.log2(K))
    catnat = CatNat(K=K, activation='sigmoid')   # sigmoid gives cleaner P(a_i) for visualization

    # Try 3 different score configurations
    configs = [
        (torch.zeros(1, K-1),                          "s=0 (uniform)"),
        (torch.tensor([[3.0, -3.0, 0.5, 0.0, 0.5, 0.0, -0.5]]),  "peaked left"),
        (torch.tensor([[-3.0, 3.0, -0.5, 0.0, -0.5, 0.0, 0.5]]), "peaked right"),
    ]

    fig, axes = plt.subplots(1, len(configs), figsize=(14, 5))

    # BFS layout for the tree (positions per level)
    # Node 0: root (h=1), nodes 1-2: h=2, nodes 3-6: h=3
    node_positions = {
        0: (0.5, 3),
        1: (0.25, 2), 2: (0.75, 2),
        3: (0.125,1), 4: (0.375,1), 5: (0.625,1), 6: (0.875,1),
    }
    edges = [(0,1),(0,2),(1,3),(1,4),(2,5),(2,6)]

    for ax, (s, title) in zip(axes, configs):
        with torch.no_grad():
            reach = catnat.node_reach_probs(s).squeeze(0).numpy()   # [K-1]
            probs = catnat(s).squeeze(0).numpy()                     # [K]

        # Draw edges
        for (pa, ch) in edges:
            xp, yp = node_positions[pa]; xc, yc = node_positions[ch]
            ax.plot([xp, xc], [yp, yc], 'k-', lw=1.5, alpha=0.4, zorder=1)

        # Draw internal nodes (color = P(a_i))
        for node_i, (xp, yp) in node_positions.items():
            r = reach[node_i]
            circle = plt.Circle((xp, yp), 0.06, color=plt.cm.Blues(0.3 + 0.7*r),
                                 ec='navy', lw=1.5, zorder=3)
            ax.add_patch(circle)
            ax.text(xp, yp, f'{r:.2f}', ha='center', va='center', fontsize=8, fontweight='bold', zorder=4)

        # Draw leaf probabilities below tree
        leaf_x = [0.0625 + i*0.125 for i in range(K)]
        for i, (lx, p) in enumerate(zip(leaf_x, probs)):
            ax.bar(lx, p*0.5, width=0.1, bottom=0.3, color=plt.cm.Oranges(0.3 + p*3),
                   ec='darkorange', alpha=0.85)
            ax.text(lx, 0.28, f'p{i}\n{p:.2f}', ha='center', fontsize=6.5)

        ax.set_xlim(-0.05, 1.05); ax.set_ylim(0.1, 3.5)
        ax.set_title(f'{title}\nNode values = P(aᵢ)', fontsize=10)
        ax.axis('off')

    # Shared colorbar
    sm = plt.cm.ScalarMappable(cmap='Blues', norm=mcolors.Normalize(0,1))
    fig.colorbar(sm, ax=axes.tolist(), shrink=0.6, label='P(aᵢ) — node reach probability')
    plt.suptitle('catnat tree (K=8): node reach probabilities and leaf probabilities\n'
                 'Each node\'s FIM entry scales with P(aᵢ) (Theorem 4.2)', fontsize=11)
    plt.show()

except Exception as e:
    print(f"[ERROR] {e}")
    raise

## What these visualizations show

**Figure 1 (loss landscape):** The catnat landscape has more regular contours and the gradient-descent trajectories (black lines) are more direct than softmax. This matches Figure 1 of the paper.

**FIM diagonal sweep:** For catnat-ν, all nodes at the same tree level have identical FIM entries (Corollary 4.3 — entries depend only on $P(a_i)$ not on $s_i$). For catnat-σ they differ within a level because $\sigma(s_i)(1-\sigma(s_i))$ varies with $s_i$.

**Gumbel ablation:** Lower temperature → samples concentrate around the mode (lower entropy). The annealing schedule maintains differentiability during training while pushing toward discrete samples at convergence.

**Tree reach probabilities:** The root is always reached with $P=1$. Deeper nodes have smaller reach probabilities determined by the ancestors' binary decisions. The FIM diagonal entries are proportional to these reach probabilities — deeper nodes (smaller $P(a_i)$) contribute less to the Fisher metric, naturally balancing the optimization across tree levels.